# PDEFlow: portable Colab launcher

This notebook calls the installed package. CUDA pilots are provided but have not been hardware-tested. CPU smoke execution was verified on macOS. A smoke run is not scientific model validation.

Select a pinned source commit **or** upload the source ZIP produced by `scripts/package_source.py`. Choose persistent output storage explicitly; Colab runtime storage and the environment/cache below are disposable. No Drive mount or account is assumed. Use Python 3.11/3.12; the tested configuration is Python 3.12.3.

Heat and static joint permeability/pressure Darcy use the same package. Choose darcy_colab_2d or darcy_colab_3d with DEVICE="cuda" only on a suitable runtime; those full profiles have not been run here. The restricted conventional Darcy inverse and a bounded negative study are implemented; see docs/darcy-study.md. These smoke checkpoints do not establish scientific accuracy.

Online/hybrid modes still solve clean numerical references, but require no saved training dataset. Fixed evaluation manifests and checkpoints persist. All three specialists share this workflow; actual Colab/CUDA execution remains untested.

The preferred everyday workflow is now the named-preset section at the end: the package automatically prepares evaluation/calibration and discovers exports. Earlier explicit-config cells are retained for advanced compatibility.


In [ ]:
from pathlib import Path
import json, os, re, subprocess, sys, tarfile, venv, zipfile

# Fill exactly one source option. No repository URL is assumed.
REPOSITORY_URL = ""
COMMIT_SHA = ""  # exact 40-character commit SHA
SOURCE_ARCHIVE = ""  # path to an uploaded .zip or .tar.gz
CACHE_ROOT = Path("/content/pdeflow-cache")  # disposable, configurable
persistent_text = input("Persistent output directory (set up your storage first): ").strip()
if not persistent_text:
    raise ValueError("Choose an explicit persistent output path")
OUTPUT_ROOT = Path(persistent_text).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError("Use a Python 3.11/3.12 runtime; tested on Python 3.12.3")


In [ ]:
SOURCE_ROOT = CACHE_ROOT / "source"
import hashlib
source_identity = {"repository": REPOSITORY_URL, "commit": COMMIT_SHA} if REPOSITORY_URL else {"archive_sha256": hashlib.sha256(Path(SOURCE_ARCHIVE).read_bytes()).hexdigest()}
identity_file = CACHE_ROOT / "source-identity.json"
if SOURCE_ROOT.exists() and (not identity_file.exists() or json.loads(identity_file.read_text()) != source_identity):
    raise ValueError("Cached source differs or is incomplete; choose a fresh CACHE_ROOT")
if bool(REPOSITORY_URL) == bool(SOURCE_ARCHIVE):
    raise ValueError("Configure exactly one source option")
if not SOURCE_ROOT.exists():
    if REPOSITORY_URL:
        if not re.fullmatch(r"[0-9a-fA-F]{40}", COMMIT_SHA):
            raise ValueError("Use an exact commit SHA for reproducibility")
        subprocess.run(["git", "clone", "--no-checkout", REPOSITORY_URL, str(SOURCE_ROOT)], check=True)
        subprocess.run(["git", "-C", str(SOURCE_ROOT), "checkout", "--detach", COMMIT_SHA], check=True)
    else:
        SOURCE_ROOT.mkdir()
        archive = Path(SOURCE_ARCHIVE).expanduser().resolve()
        if zipfile.is_zipfile(archive):
            with zipfile.ZipFile(archive) as z:
                for member in z.infolist():
                    target = (SOURCE_ROOT / member.filename).resolve()
                    if not target.is_relative_to(SOURCE_ROOT.resolve()):
                        raise ValueError("Unsafe archive path")
                z.extractall(SOURCE_ROOT)
        else:
            with tarfile.open(archive) as t:
                # Python 3.11.8+ / 3.12 safe extraction filter.
                t.extractall(SOURCE_ROOT, filter="data")
if REPOSITORY_URL:
    checked = subprocess.check_output(["git", "-C", str(SOURCE_ROOT), "rev-parse", "HEAD"], text=True).strip()
    if checked.lower() != COMMIT_SHA.lower():
        raise ValueError("Cached checkout does not match the pinned commit")
identity_file.write_text(json.dumps(source_identity))
roots = list(SOURCE_ROOT.glob("pyproject.toml")) or list(SOURCE_ROOT.glob("*/pyproject.toml"))
if len(roots) != 1:
    raise ValueError("Expected one package source root")
PACKAGE_ROOT = roots[0].parent
ENV_ROOT = CACHE_ROOT / "env"
if not ENV_ROOT.exists():
    venv.EnvBuilder(with_pip=True).create(ENV_ROOT)
PYTHON = ENV_ROOT / "bin/python"
subprocess.run([str(PYTHON), "-m", "pip", "install", "-c", str(PACKAGE_ROOT / "requirements-tested.txt"),
                "-e", str(PACKAGE_ROOT) + "[test]"], check=True)

def cli(*args):
    subprocess.run([str(PYTHON), "-m", "pdeflow.cli", *map(str, args)], check=True)

cli("hardware")
# The kernel's preinstalled torch is irrelevant: package runs in this isolated environment.


In [ ]:
PROFILE = "cpu_2d"  # also online_heat_2d, online_darcy_2d, online_chemistry_2d, online_colab_chemistry_3d
DEVICE = "cpu"  # cuda for a GPU pilot; no concealed fallback
RUN_NAME = PROFILE + "-run1"  # choose a new name for an independent run
RUN_ROOT = OUTPUT_ROOT / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)
config = json.loads((PACKAGE_ROOT / "configs/experiments" / (PROFILE + ".json")).read_text())
config["runtime"]["device"] = DEVICE
CONFIG_PATH = RUN_ROOT / "config.json"
if CONFIG_PATH.exists() and json.loads(CONFIG_PATH.read_text()) != config:
    raise ValueError("Existing run uses another configuration; select a new run name")
CONFIG_PATH.write_text(json.dumps(config, indent=2))
cli("validate-config", "--config", CONFIG_PATH)


In [ ]:
# Persist fixed evaluation and optional offline data; runtime caches are disposable.
import shutil
provider = config.get("provider")
DATA = CACHE_ROOT / RUN_NAME / "data"
PERSISTENT_DATA = RUN_ROOT / "data"
EVALUATION = CACHE_ROOT / RUN_NAME / "evaluation"
PERSISTENT_EVALUATION = RUN_ROOT / "evaluation"
TRAIN = RUN_ROOT / "training"
command = ["train", "--config", CONFIG_PATH, "--output", TRAIN]
if provider:
    if not (EVALUATION / "manifest.json").exists() and (PERSISTENT_EVALUATION / "manifest.json").exists():
        shutil.copytree(PERSISTENT_EVALUATION, EVALUATION)
    if not (EVALUATION / "manifest.json").exists():
        cli("prepare-evaluation", "--config", CONFIG_PATH, "--output", EVALUATION)
    if not PERSISTENT_EVALUATION.exists():
        shutil.copytree(EVALUATION, PERSISTENT_EVALUATION)
    command += ["--evaluation", EVALUATION]
    if provider["mode"] == "offline":
        if not (DATA / "manifest.json").exists() and PERSISTENT_DATA.exists():
            shutil.copytree(PERSISTENT_DATA, DATA)
        if not (DATA / "manifest.json").exists():
            cli("prepare-training", "--config", CONFIG_PATH, "--output", DATA)
        if not PERSISTENT_DATA.exists():
            shutil.copytree(DATA, PERSISTENT_DATA)
        command += ["--data", DATA]
    PROBLEM = EVALUATION / "test/0/problem.json"
    OBSERVATIONS = EVALUATION / "test/0/observations.json"
    REFERENCE = EVALUATION / "test/0/reference.npz"
else:
    if not (DATA / "manifest.json").exists() and PERSISTENT_DATA.exists():
        shutil.copytree(PERSISTENT_DATA, DATA)
    if not (DATA / "manifest.json").exists():
        cli("generate", "--config", CONFIG_PATH, "--output", DATA)
    if not PERSISTENT_DATA.exists():
        shutil.copytree(DATA, PERSISTENT_DATA)
    command += ["--data", DATA]
    PROBLEM = DATA / "test_problem.json"
    OBSERVATIONS = DATA / "test_observations.json"
    REFERENCE = DATA / "test_reference.npz"
if (TRAIN / "checkpoint.pt").exists():
    command += ["--resume", TRAIN / "checkpoint.pt"]
cli(*command)
# Re-run this cell with identical source/config/manifests to resume safely.
cli("inspect-checkpoint", "--checkpoint", TRAIN / "inference.pt")
EXPORT = RUN_ROOT / "portable-export"
EXPORT.mkdir(exist_ok=True)
for source, name in [(TRAIN / "inference.pt", "inference.pt"), (CONFIG_PATH, "config.json"),
                     (PROBLEM, "test_problem.json"), (OBSERVATIONS, "test_observations.json")]:
    shutil.copy2(source, EXPORT / name)


In [ ]:
RECON = RUN_ROOT / "reconstruction"
if not (RECON / "comparison.json").exists():
    cli("reconstruct", "--config", CONFIG_PATH, "--checkpoint", TRAIN / "inference.pt",
        "--problem", PROBLEM, "--observations", OBSERVATIONS,
        "--reference", REFERENCE, "--output", RECON)
cli("evaluate", "--result", RECON / "observation_pde")
from IPython.display import display, Image
display(Image(filename=str(RECON / "observation_pde/reconstruction.png")))
print("Portable inference checkpoint:", TRAIN / "inference.pt")
print("Copy it with config.json, test_problem.json and test_observations.json to your Mac.")


## Local CPU reconstruction

Install the same pinned source and dependencies locally. With downloaded files in a chosen directory:

```bash
pdeflow reconstruct --config config.json --device cpu --checkpoint inference.pt \
  --problem test_problem.json --observations test_observations.json --output local-reconstruction
```

No training dataset or hidden reference is required. Supply `--reference test_reference.npz` only to compute held-out error. Keep the full `checkpoint.pt` for training resume. Actual Colab/CUDA execution and CUDA-to-MPS transfer remain untested. Local CPU/MPS preflights are recorded separately; cross-device bitwise equality is not promised. Do not use the small smoke results as evidence of posterior calibration or scientific reconstruction quality.


## Simplified persistent train/resume (preferred)

Use the same package commands as on the Mac. Choose the persistent workspace explicitly. This cell launches real training when run; CUDA/actual Colab are untested. The larger colab-heat-2d profile is configured, not hardware validated. For a short local check choose quickstart-heat-2d and cpu. Do not run both this section and the earlier advanced workflow unless you intend separate runs.


In [ ]:
LOCAL_WORKSPACE = OUTPUT_ROOT / "local-workspace"  # persistent, configurable
LOCAL_PRESET = "colab-heat-2d"
LOCAL_DEVICE = "cuda"  # explicit; never silently falls back
managed_runs = list((LOCAL_WORKSPACE / "runs").glob("*/run.json"))
if managed_runs:
    cli("resume", "latest", "--workspace", LOCAL_WORKSPACE, "--device", LOCAL_DEVICE)
else:
    cli("train", "--preset", LOCAL_PRESET, "--workspace", LOCAL_WORKSPACE, "--device", LOCAL_DEVICE, "--plain")
cli("status", "latest", "--workspace", LOCAL_WORKSPACE)


In [ ]:
# One recorded validation case; explicit --split test opts into held-out test inspection.
cli("evaluate", "--run", "latest", "--workspace", LOCAL_WORKSPACE, "--device", LOCAL_DEVICE)


To extend the recorded total budget, use cli("resume", "latest", "--workspace", LOCAL_WORKSPACE, "--additional-steps", "1000"). Constant Adam learning rate and optimizer state are retained. Transfer the whole managed workspace for status/evaluate/resume, or just training/inference.pt (latest) / training/best-inference.pt (minimum validation CFM objective) for portable inference. On the Mac run pdeflow models --import PATH to link exports without moving them. Use an explicit CPU/MPS runtime for inference; training resume requires the recorded source/config and does not promise cross-device bitwise identity. No test-based checkpoint selection occurs.
